# 02d — Scenario A/B Definitions from trials_scenarios

This companion notebook defines concrete scenario slices on top of the
canonical Phase 2 scenario table, without modifying the original
`02_scenario_prep_and_risk_features.ipynb` pipeline.

We start from `data/interim/trials_scenarios.parquet`, which now includes:

- One row per `nct_id`
- Trial context: `phase`, `overall_status`, `brief_title`
- Sponsor / region: `lead_sponsor`, `lead_sponsor_norm`, `region_label`
- Operational features: `estimated_trial_cost`, `enrollment_feasibility_score`
- Heuristic benefit: `benefit_score` (from `02c_benefit_scoring_and_scenarios.ipynb`)

In this notebook we define two illustrative scenarios:

- **Scenario A — High-benefit, active mid/late-phase trials**
  - `phase` in {Phase 2, Phase 2/Phase 3, Phase 3}
  - `overall_status` ∈ {Recruiting, Enrolling by invitation, Active, not recruiting}
  - `benefit_score` ≥ 0.5

- **Scenario B — High-benefit, cost-conscious trials**
  - Start from Scenario A
  - Keep only trials with `estimated_trial_cost` ≤ median cost of Scenario A

For each scenario we create a filtered table and persist it under `data/scenarios/`
for downstream optimization notebooks.


In [1]:
# ============================================================
# Cell 1 — Load canonical trials_scenarios table
# ============================================================

from pathlib import Path
import pandas as pd

def log(msg: str) -> None:
    print(msg)

SCENARIOS_PARQUET = Path("data/interim/trials_scenarios.parquet")

if SCENARIOS_PARQUET.exists():
    trials_scenarios = pd.read_parquet(SCENARIOS_PARQUET)
    log(f"[Cell 1] Loaded trials_scenarios with shape {trials_scenarios.shape}")
else:
    raise FileNotFoundError(
        f"[Cell 1] Missing {SCENARIOS_PARQUET}. "
        "Run Phase 2 notebooks (02 and 02c) first."
    )

trials_scenarios.head()

[Cell 1] Loaded trials_scenarios with shape (557292, 13)


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score
0,NCT00000102,Congenital Adrenal Hyperplasia: Calcium Channe...,Completed,Phase 1/Phase 2,[Congenital Adrenal Hyperplasia],[Nifedipine],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.5,1.0,0.27
1,NCT00000104,Does Lead Burden Alter Neuropsychological Deve...,Completed,None,[Lead Poisoning],[ERP measures of attention and memory],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0,0.18
2,NCT00000105,Vaccination With Tetanus and KLH to Assess Imm...,Terminated,None,[Cancer],"[Intracel KLH Vaccine, Biosyn KLH, Montanide I...",[United States],"Masonic Cancer Center, University of Minnesota","Masonic Cancer Center, University of Minnesota",Global / Multi-Region,1.0,1.0,0.06
3,NCT00000106,41.8 Degree Centigrade Whole Body Hyperthermia...,Unknown status,N/A,[Rheumatic Diseases],[Whole body hyperthermia unit],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0,0.08
4,NCT00000107,Body Water Content in Cyanotic Congenital Hear...,Completed,None,"[Heart Defects, Congenital]",[],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0,0.18


### What Cell 1 Just Did

This step loaded the canonical Phase 2 scenario table from
`data/interim/trials_scenarios.parquet` into `trials_scenarios` and previewed
the first few rows.

At this point, `trials_scenarios` contains one row per `nct_id` with the full
set of scenario features, including `phase`, `overall_status`,
`estimated_trial_cost`, `enrollment_feasibility_score`, and the heuristic
`benefit_score` introduced in `02c_benefit_scoring_and_scenarios.ipynb`.

The rest of this notebook will define Scenario A/B as simple filtered views of
this table.


In [2]:
# ============================================================
# Cell 2 — Define Scenario A and Scenario B
# ============================================================
#
# Scenario A — High-benefit, active mid/late-phase trials
#   - phase in {"Phase 2", "Phase 2/Phase 3", "Phase 3"}
#   - overall_status in {"Recruiting", "Enrolling by invitation", "Active, not recruiting"}
#   - benefit_score >= 0.5
#
# Scenario B — High-benefit, cost-conscious trials
#   - Subset of Scenario A
#   - estimated_trial_cost <= median(cost in Scenario A)

import numpy as np

required_cols = [
    "nct_id",
    "phase",
    "overall_status",
    "benefit_score",
    "estimated_trial_cost",
]

missing = [c for c in required_cols if c not in trials_scenarios.columns]
if missing:
    log(f"[Cell 2] ERROR: Missing required columns in trials_scenarios: {missing}")
else:
    # --- Define filters for Scenario A ------------------------------------
    mid_late_phases = {"Phase 2", "Phase 2/Phase 3", "Phase 3"}
    active_statuses = {
        "Recruiting",
        "Enrolling by invitation",
        "Active, not recruiting",
    }

    mask_phase = trials_scenarios["phase"].isin(mid_late_phases)
    mask_status = trials_scenarios["overall_status"].isin(active_statuses)
    mask_benefit = trials_scenarios["benefit_score"] >= 0.5

    scenario_A_mask = mask_phase & mask_status & mask_benefit
    scenario_A = trials_scenarios.loc[scenario_A_mask].copy()

    log(f"[Cell 2] Scenario A size: {scenario_A.shape[0]:,} trials")

    if scenario_A.empty:
        log("[Cell 2] WARNING: Scenario A is empty; Scenario B will also be empty.")
        scenario_B = scenario_A.copy()
    else:
        # --- Define Scenario B as cost-conscious subset of A --------------
        median_cost_A = scenario_A["estimated_trial_cost"].median()
        log(f"[Cell 2] Scenario A median estimated_trial_cost: {median_cost_A:,.2f}")

        scenario_B_mask = scenario_A["estimated_trial_cost"] <= median_cost_A
        scenario_B = scenario_A.loc[scenario_B_mask].copy()

        log(f"[Cell 2] Scenario B size: {scenario_B.shape[0]:,} trials")

    # Peek at a few rows from each scenario
    log("[Cell 2] Scenario A preview:")
    display(
        scenario_A[
            ["nct_id", "phase", "overall_status", "estimated_trial_cost", "benefit_score"]
        ].head(10)
    )

    log("[Cell 2] Scenario B preview:")
    display(
        scenario_B[
            ["nct_id", "phase", "overall_status", "estimated_trial_cost", "benefit_score"]
        ].head(10)
    )


[Cell 2] Scenario A size: 16,672 trials
[Cell 2] Scenario A median estimated_trial_cost: 2.00
[Cell 2] Scenario B size: 10,268 trials
[Cell 2] Scenario A preview:


,nct_id,phase,overall_status,estimated_trial_cost,benefit_score
2153,NCT00002462,Phase 3,"Active, not recruiting",4.0,0.8
2630,NCT00003042,Phase 2,"Active, not recruiting",2.0,0.6
3144,NCT00003641,Phase 3,"Active, not recruiting",4.0,0.8
4672,NCT00005584,Phase 3,"Active, not recruiting",4.0,0.8
5561,NCT00006721,Phase 3,"Active, not recruiting",4.0,0.8
5590,NCT00007150,Phase 2,"Active, not recruiting",2.0,0.6
6300,NCT00018057,Phase 2,Recruiting,2.0,0.6
6855,NCT00026312,Phase 3,"Active, not recruiting",4.0,0.8
8127,NCT00044304,Phase 2,Recruiting,2.0,0.6
9932,NCT00070499,Phase 2,"Active, not recruiting",2.0,0.6


[Cell 2] Scenario B preview:


,nct_id,phase,overall_status,estimated_trial_cost,benefit_score
2630,NCT00003042,Phase 2,"Active, not recruiting",2.0,0.6
5590,NCT00007150,Phase 2,"Active, not recruiting",2.0,0.6
6300,NCT00018057,Phase 2,Recruiting,2.0,0.6
8127,NCT00044304,Phase 2,Recruiting,2.0,0.6
9932,NCT00070499,Phase 2,"Active, not recruiting",2.0,0.6
10343,NCT00076310,Phase 2,"Active, not recruiting",2.0,0.6
10411,NCT00077285,Phase 2,"Active, not recruiting",2.0,0.6
10655,NCT00080756,Phase 2,"Active, not recruiting",2.0,0.6
10788,NCT00082706,Phase 2,"Active, not recruiting",2.0,0.6
11008,NCT00085982,Phase 2,"Active, not recruiting",2.0,0.6


### What Cell 2 Just Did

This step turned the raw `trials_scenarios` table into two concrete scenario
slices:

- **Scenario A — High-benefit, active mid/late-phase trials**
  - Kept trials where:
    - `phase` is one of {Phase 2, Phase 2/Phase 3, Phase 3}
    - `overall_status` is one of:
      - Recruiting
      - Enrolling by invitation
      - Active, not recruiting
    - `benefit_score` ≥ 0.5

- **Scenario B — High-benefit, cost-conscious trials**
  - Started from Scenario A.
  - Computed the median `estimated_trial_cost` within Scenario A.
  - Kept only trials with `estimated_trial_cost` less than or equal
    to this median, yielding a lower-cost subset of A.

The cell then logged the number of trials in each scenario and displayed a
small preview of both slices, showing `nct_id`, `phase`, `overall_status`,
`estimated_trial_cost`, and `benefit_score`.


In [4]:
# ============================================================
# Cell 3 — Persist Scenario A/B tables to data/scenarios/
# ============================================================

SCENARIO_DIR = Path("data/scenarios")
SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

scenario_A_path = SCENARIO_DIR / "scenario_A_trials.csv"
scenario_B_path = SCENARIO_DIR / "scenario_B_trials.csv"

if "scenario_A" in globals() and "scenario_B" in globals():
    # Save richer tables (not just IDs) so we can filter directly later.
    scenario_A.to_csv(scenario_A_path, index=False)
    scenario_B.to_csv(scenario_B_path, index=False)

    log(f"[Cell 3] Wrote Scenario A to {scenario_A_path} with shape {scenario_A.shape}")
    log(f"[Cell 3] Wrote Scenario B to {scenario_B_path} with shape {scenario_B.shape}")
else:
    log("[Cell 3] ERROR: scenario_A / scenario_B not found; nothing was written.")


[Cell 3] Wrote Scenario A to data/scenarios/scenario_A_trials.csv with shape (16672, 13)
[Cell 3] Wrote Scenario B to data/scenarios/scenario_B_trials.csv with shape (10268, 13)


### What Cell 3 Just Did

This step materialized the in-memory scenario slices as reusable artifacts under
`data/scenarios/`:

- `data/scenarios/scenario_A_trials.csv`
- `data/scenarios/scenario_B_trials.csv`

Rather than writing only `nct_id` values, the notebook saved the **full set of
columns** for each scenario, including sponsor, region, cost, feasibility, and
benefit. This makes it easy for downstream optimization notebooks to:

- Look up scenario-specific trial subsets, and
- Further filter or rank within a scenario without needing to rejoin to the
  main `trials_scenarios` table.


## Notebook Summary — Scenario A/B Definitions

This helper notebook defined two concrete, reproducible scenario slices on top
of the canonical Phase 2 scenario table:

1. **Scenario A — High-benefit, active mid/late-phase trials**
   - Phase 2 / 2–3 / 3
   - Recruiting or otherwise active
   - Heuristic `benefit_score` ≥ 0.5

2. **Scenario B — High-benefit, cost-conscious trials**
   - Subset of Scenario A
   - `estimated_trial_cost` at or below the Scenario A median

Both scenarios were saved as CSV tables under `data/scenarios/`, providing a
clear bridge between the rich feature layer in `trials_scenarios` and the
trial subsets that future optimization and quantum notebooks will operate on.

The original Phase 2 feature-building pipeline remains unchanged; this notebook
adds only scenario definitions and artifacts on top of it.
